# train_commented.ipynb — анотована копія train.ipynb

Цей блокнот **повністю еквівалентний** train.ipynb (можна запустити так само).
Кожен рядок коду має поряд коментар українською, що пояснює, що він робить.

Призначений як навчальний матеріал — для розуміння тренувального пайплайну
без потреби паралельно читати окремий документ-розбір.


## Клітинка 1. Імпорти та налаштування

Імпортуємо всі необхідні бібліотеки, визначаємо шляхи до директорій,
вмикаємо/вимикаємо великі операції через прапорці-перемикачі.


In [ ]:
# ─── Імпорти зі стандартної бібліотеки ───────────────────────────────────
import os, gc, json, time, resource     # os: змінні середовища; gc: garbage collector; json: для cfg
                                         # time: таймінги; resource: для отримання RSS пам'яті процесу
from pathlib import Path                 # Path: об'єктно-орієнтований шлях замість сирих рядків

# ─── Імпорти зовнішніх пакетів ───────────────────────────────────────────
import numpy as np                       # numpy — числові масиви, базова бібліотека
import pandas as pd                      # pandas — DataFrame для агрегованих метрик
import matplotlib.pyplot as plt          # matplotlib — графіки для EDA і curves
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 — реєструє 3D-проекцію для add_subplot

import trimesh                           # trimesh — робота з 3D-mesh (.obj/.stl)
import pysdf                             # pysdf — швидкий C++ обчислювач SDF (AABB-tree)
import torch                             # PyTorch — для моделі MultiResHashSDF

# ─── Наш модуль ──────────────────────────────────────────────────────────
from sdf_model import MultiResHashSDF, HashConfig   # модель і її конфігурація з sdf_model.py

# ─── Шляхи ───────────────────────────────────────────────────────────────
MESH_DIR   = Path('test_task_meshes')                # директорія з 50-ма .obj файлами
MODELS_DIR = Path('models'); MODELS_DIR.mkdir(exist_ok=True)  # директорія для збереження ваг
                                                              # exist_ok=True — не помилка якщо існує

# ─── Прапорці-перемикачі важких операцій ─────────────────────────────────
# Безпекова механіка: жодна важка операція не запуститься автоматично.
# Користувач має явно встановити True для тієї, яку хоче запустити.
RUN_TRAIN_ALL    = True       # тренування всіх 50 mesh-ів (~40-60 хв на M4 CPU)
RUN_EVAL_ALL     = True       # обчислення F1-метрик на свіжому eval-наборі
RUN_RETRAIN_HARD = False      # перетренування важких mesh-ів з більшою T (~15 хв)
RUN_BENCH        = True       # бенчмарк через numba inference

# ─── Пристрій для PyTorch ────────────────────────────────────────────────
# MPS (Apple Metal) має великий dispatch-overhead для маленьких моделей,
# тому CPU фактично швидший. Якщо є NVIDIA GPU, можна замінити на 'cuda'.
DEVICE = torch.device('cpu')

# ─── Утиліта для RSS-пам'яті процесу ─────────────────────────────────────
def mem_mb() -> float:
    """Повертає resident memory size процесу в МБ."""
    rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    # macOS reports bytes, Linux reports KB — конвертуємо до МБ.
    return rss / (1024**2) if os.uname().sysname == 'Darwin' else rss / 1024

# ─── Список усіх mesh-файлів ─────────────────────────────────────────────
# .glob('*.obj') — generator усіх .obj файлів у директорії.
# key=lambda p: int(p.stem) — сортуємо по числовому імені (щоб '10.obj' йшов після '2.obj').
# p.stem — ім'я файлу без розширення.
mesh_files = sorted(MESH_DIR.glob('*.obj'), key=lambda p: int(p.stem))
print(f'Found {len(mesh_files)} meshes  |  device: {DEVICE}  |  RSS: {mem_mb():.0f} MB')


## Клітинка 2. Утиліти для роботи з mesh

Дві простi функції: завантаження mesh-у з .obj та його нормалізація у куб [-0.95, 0.95]³.


In [ ]:
def load_mesh(path: Path) -> trimesh.Trimesh:
    """Завантажує .obj файл і повертає Trimesh-об'єкт."""
    # force='mesh' — каже trimesh повернути Trimesh (а не Scene з кількома об'єктами).
    m = trimesh.load(path, force='mesh')
    # Іноді trimesh все одно повертає Scene попри force='mesh'. Об'єднуємо вручну.
    if isinstance(m, trimesh.Scene):
        m = trimesh.util.concatenate([
            trimesh.Trimesh(vertices=g.vertices, faces=g.faces) for g in m.geometry.values()])
    return m

def normalize_mesh(m: trimesh.Trimesh, padding: float = 0.95) -> trimesh.Trimesh:
    """Центрує mesh у нулі, ізотропно масштабує так, щоб bbox був у [-padding, padding]."""
    m = m.copy()                                           # копія, щоб не модифікувати оригінал
    center = (m.bounds[0] + m.bounds[1]) / 2.0             # геометричний центр bbox = (min + max) / 2
    m.apply_translation(-center)                           # переносимо center у (0, 0, 0)
    # np.abs(m.bounds).max() — найдальша координата по модулю. Для центрованого mesh = напіврозмах.
    # padding / max_abs — коефіцієнт масштабування. Якщо padding=0.95, mesh потрапить у [-0.95, 0.95]³.
    m.apply_scale(padding / np.abs(m.bounds).max())
    return m

print('mesh utilities defined.')


## Клітинка 3. EDA — exploratory data analysis

Збираємо статистики по всіх 50 mesh-ах: кількість вершин/трикутників,
чи вони закриті, об'єм, площу. Це допомагає зрозуміти датасет та виявити
проблемні випадки (наприклад, mesh-і з нульовим внутрішнім об'ємом).


In [ ]:
rows = []                                                  # збиратимемо статистику в список словників
for p in mesh_files:                                       # цикл по всіх 50 mesh-ах
    m = load_mesh(p)                                       # завантажуємо mesh
    bbox = m.bounds; extents = bbox[1] - bbox[0]           # bbox = [[min], [max]]; extents = розміри по осях
    rows.append({
        'id': int(p.stem),                                 # числовий ID файлу
        'n_vertices': len(m.vertices),                     # кількість вершин у mesh
        'n_faces': len(m.faces),                           # кількість трикутників
        'watertight': bool(m.is_watertight),               # чи mesh "водотривкий" (без дірок)
        'area': float(m.area),                             # площа поверхні
        # Об'єм визначений тільки для watertight mesh; інакше NaN
        'volume': float(m.volume) if m.is_watertight else float('nan'),
        'max_extent': float(extents.max()),                # найдовша вісь
        # volume_frac = частка bbox, яку займає mesh (індикатор "товщини")
        'volume_frac': float(m.volume) / (extents[0]*extents[1]*extents[2]) if m.is_watertight else float('nan'),
    })
    del m; gc.collect()                                    # явно прибираємо mesh, щоб звільнити пам'ять

# Конструюємо pandas DataFrame і сортуємо по ID
df = pd.DataFrame(rows).sort_values('id').reset_index(drop=True)
print(f'Watertight: {df["watertight"].sum()}/{len(df)}  |  RSS: {mem_mb():.0f} MB')
df.describe()                                              # описова статистика всіх колонок


## Клітинка 4. Гістограми ключових характеристик


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3))            # 3 субплоти в одному рядку
axes[0].hist(df['n_faces'], bins=25);     axes[0].set_title('#faces')           # розподіл #трикутників
axes[1].hist(df['volume_frac'].dropna(), bins=25); axes[1].set_title('volume / bbox vol')  # частка bbox
axes[2].hist(df['area'], bins=25);        axes[2].set_title('surface area')    # площі поверхні
plt.tight_layout(); plt.show()                             # автоматичне розташування + показ


## Клітинка 5. Візуалізація 6 нормалізованих mesh-ів


In [ ]:
fig = plt.figure(figsize=(12, 8))                          # велике полотно
for i, mid in enumerate([0, 5, 10, 18, 26, 49]):           # 6 mesh-ів для огляду
    m = normalize_mesh(load_mesh(MESH_DIR / f'{mid}.obj')) # завантажуємо і нормалізуємо
    pts, _ = trimesh.sample.sample_surface(m, 1500)        # семплимо 1500 точок з поверхні
    ax = fig.add_subplot(2, 3, i+1, projection='3d')       # 2x3 сітка субплотів, 3D-проекція
    ax.scatter(pts[:,0], pts[:,1], pts[:,2], s=0.5)        # scatter точок (маленькі)
    ax.set_title(f'mesh {mid}  ({len(m.faces)} faces)')
    ax.set_xlim(-1,1); ax.set_ylim(-1,1); ax.set_zlim(-1,1)  # стандартний домен
    ax.set_box_aspect((1,1,1))                              # рівні масштаби — куб виглядає як куб
    del m, pts; gc.collect()
plt.tight_layout(); plt.show()


## Клітинка 6. Допоміжна функція F1 та основна функція тренування

`compute_f1` — обчислює F1-score для бінарної класифікації (всередині vs зовні).

`train_one_mesh` — повний цикл тренування для одного mesh-у з online resampling.


In [ ]:
def compute_f1(pred: np.ndarray, gt_inside: np.ndarray) -> float:
    """F1-score для класифікації occupancy. Inside = positive class."""
    pred_inside = pred < 0                                  # передбачили "всередині" якщо SDF < 0
    tp = np.logical_and(pred_inside,  gt_inside).sum()      # TP: правильно передбачили "всередині"
    fp = np.logical_and(pred_inside, ~gt_inside).sum()      # FP: передбачили "всередині", насправді зовні
    fn = np.logical_and(~pred_inside, gt_inside).sum()      # FN: передбачили "зовні", насправді всередині
    if tp + fp + fn == 0:                                   # обидва класи порожні → ідеальна класифікація
        return 1.0
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1) # max(., 1) — захист від ділення на 0
    return 0.0 if prec + rec == 0 else float(2 * prec * rec / (prec + rec))  # F1 = 2PR / (P+R)

def train_one_mesh(mid: int,
                   n_iters: int = 2500,                     # максимум ітерацій (можливий ранній стоп)
                   cfg: HashConfig = None,                  # None → дефолтна конфігурація
                   lr: float = 5e-3,                        # learning rate для Adam
                   per_step=dict(surface=512, near_1e2=4000, near_1e3=2000, uniform=1024),  # квоти batch
                   eval_every: int = 250,                   # частота eval (кожні 250 ітерацій)
                   early_stop_f1: float = 0.96,             # поріг F1 для раннього зупинення
                   verbose: bool = False):
    """Тренер з online resampling для одного mesh. Повертає (model, history)."""
    # ─── Завантаження mesh і створення SDF-функції ────────────────────
    mesh = normalize_mesh(load_mesh(MESH_DIR / f'{mid}.obj'))
    sdf_fn = pysdf.SDF(mesh.vertices, mesh.faces)           # pysdf будує AABB-tree один раз

    # ─── Eval-набір (незалежний RNG-seed для перевірки узагальнення) ──
    rng_e = np.random.default_rng(999 + mid); n_e = 10_000
    base_e = trimesh.sample.sample_surface(mesh, n_e)[0]    # 10k точок з поверхні
    eval_pts = np.concatenate([
        # Перші 10k — near-surface з шумом σ=1e-2 (це домен метрики F1_near)
        (base_e + rng_e.normal(0, 1e-2, (n_e, 3))).astype(np.float32),
        # Наступні 10k — рівномірно з куба [-1,1]³ (для F1_unif)
        rng_e.uniform(-1, 1, (n_e, 3)).astype(np.float32)], axis=0)
    eval_sdf = (-sdf_fn(eval_pts)).astype(np.float32)       # -1 * pysdf → стандартна конвенція (neg=inside)
    eval_gi_near = eval_sdf[:n_e] < 0                       # ground truth: всередині для near-набору
    eval_gi_unif = eval_sdf[n_e:] < 0                       # ground truth: всередині для unif-набору
    eval_pts_t = torch.from_numpy(eval_pts).to(DEVICE)      # переводимо в torch-тензор один раз

    # ─── Створення моделі та оптимізатора ─────────────────────────────
    torch.manual_seed(mid)                                  # відтворюваність ваг
    model = MultiResHashSDF(cfg if cfg is not None else HashConfig()).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)       # Adam з єдиним lr для всіх параметрів

    # ─── Основний цикл тренування ─────────────────────────────────────
    rng = np.random.default_rng(42 + mid)                   # окремий RNG (відрізняється від eval)
    Q = per_step; history = []
    for it in range(n_iters):
        # Online sampling: свіжі точки кожен крок (запобігає перенавчанню).
        base = trimesh.sample.sample_surface(mesh, Q['surface'] + Q['near_1e2'] + Q['near_1e3'])[0]
        p_s  = base[:Q['surface']]                          # точки на поверхні (без шуму)
        # Шум σ=1e-2 матчить eval-метрику; це найважливіший bucket
        p_n2 = base[Q['surface']:Q['surface']+Q['near_1e2']] + rng.normal(0, 1e-2, (Q['near_1e2'], 3))
        # Дрібніший шум σ=1e-3 — для дрібніших деталей
        p_n3 = base[-Q['near_1e3']:] + rng.normal(0, 1e-3, (Q['near_1e3'], 3))
        p_u  = rng.uniform(-1, 1, (Q['uniform'], 3))        # рівномірно у кубі
        pts_np = np.concatenate([p_s, p_n2, p_n3, p_u], axis=0).astype(np.float32)
        sdf_np = (-sdf_fn(pts_np)).astype(np.float32)       # обчислюємо GT SDF (з перевертанням знаку)

        # Захисний фільтр: pysdf зрідка повертає сміттєвий ~1e19 для точок на ребрах трикутників.
        # |sdf| < 5 — гарантовано легітимні значення (діагональ нашого куба ≈ 1.73).
        good = np.abs(sdf_np) < 5.0
        pts_np, sdf_np = pts_np[good], sdf_np[good]

        # ─── Forward + backward + step ─────────────────────────────────
        p = torch.from_numpy(pts_np).to(DEVICE); s = torch.from_numpy(sdf_np).to(DEVICE)
        pred = model(p)                                     # forward pass через хеш-сітку + MLP
        loss = (pred - s).abs().mean()                      # L1 loss (mean absolute error)
        # set_to_none=True — швидше за заповнення нулями; PyTorch розпізнає None як "чисто".
        opt.zero_grad(set_to_none=True)
        loss.backward()                                     # обчислюємо градієнти через autograd
        opt.step()                                          # оновлюємо ваги по Adam-правилу

        # ─── Періодична оцінка на held-out + раннє зупинення ───────────
        if (it + 1) % eval_every == 0 or it == n_iters - 1:
            with torch.no_grad():                           # без накопичення градієнтів — економимо пам'ять
                pe = model(eval_pts_t).cpu().numpy()        # forward на eval-наборі
            f1n = compute_f1(pe[:n_e], eval_gi_near)        # F1 для near-surface
            f1u = compute_f1(pe[n_e:], eval_gi_unif)        # F1 для uniform-bbox
            history.append({'it': it+1, 'loss': float(loss), 'f1_near': f1n, 'f1_unif': f1u})
            if verbose:
                print(f'  it={it+1:>4}  loss={float(loss):.4f}  f1_near={f1n:.3f}  f1_unif={f1u:.3f}')
            # Раннє зупинення: якщо обидва F1 ≥ порогу — модель готова.
            if f1n >= early_stop_f1 and f1u >= early_stop_f1:
                if verbose: print(f'  early stop at it={it+1}')
                break

    return model, history

print('train_one_mesh defined.')


## Клітинка 7. Тренування всіх 50 mesh-ів (гейтинг через `RUN_TRAIN_ALL`)


In [ ]:
if RUN_TRAIN_ALL:
    log = []; grand_t0 = time.time()
    for p in mesh_files:                                    # цикл по всіх mesh
        mid = int(p.stem)
        out = MODELS_DIR / f'{mid}.npz'
        # Idempotent: якщо модель вже навчена — пропускаємо. Зручно для перезапуску.
        if out.exists():
            print(f'[{mid:>2}] cached — skipping'); continue
        t0 = time.time()
        model, history = train_one_mesh(mid, verbose=False) # тренуємо
        model.export_npz(out)                               # експортуємо ваги у .npz
        size_kb = out.stat().st_size / 1024                 # розмір файлу в КБ
        last = history[-1] if history else {'f1_near': float('nan'), 'f1_unif': float('nan')}
        # Лог рядок з результатами останньої eval-ітерації
        print(f'[{mid:>2}] {time.time()-t0:5.1f}s  size={size_kb:6.1f} KB  '
              f'f1_near={last["f1_near"]:.3f}  f1_unif={last["f1_unif"]:.3f}  RSS={mem_mb():.0f} MB')
        log.append({'id': mid, **last, 'size_kb': size_kb}) # **last розпаковує словник як kwargs
        del model; gc.collect()                             # звільняємо пам'ять після кожного mesh
    pd.DataFrame(log).to_csv(MODELS_DIR / 'train_log.csv', index=False)  # зберігаємо лог
    print(f'\nTotal training time: {(time.time()-grand_t0)/60:.1f} min')
else:
    print('Full training skipped. Set RUN_TRAIN_ALL = True to enable.')


## Клітинка 8. Оцінка всіх 50 mesh-ів на свіжому eval-наборі

Принципова відмінність від eval всередині `train_one_mesh`: використовуємо
**інший RNG-seed** (7777 vs 999), щоб eval-точки не співпадали з тими,
на яких модель калібрувала ранній стоп. Це справжня перевірка узагальнення.


In [ ]:
if RUN_EVAL_ALL:
    rows = []
    for p in mesh_files:
        mid = int(p.stem)
        npz = MODELS_DIR / f'{mid}.npz'
        if not npz.exists():
            print(f'[{mid:>2}] no model — skipping'); continue

        # Готуємо незалежний eval-набір
        mesh = normalize_mesh(load_mesh(MESH_DIR / f'{mid}.obj'))
        sdf_fn = pysdf.SDF(mesh.vertices, mesh.faces)
        rng_e = np.random.default_rng(7777 + mid); n_e = 10_000  # ІНШИЙ seed, ніж у тренуванні
        base_e = trimesh.sample.sample_surface(mesh, n_e)[0]
        eval_pts = np.concatenate([
            (base_e + rng_e.normal(0, 1e-2, (n_e, 3))).astype(np.float32),
            rng_e.uniform(-1, 1, (n_e, 3)).astype(np.float32)])
        eval_sdf = (-sdf_fn(eval_pts)).astype(np.float32)

        # Завантажуємо модель і робимо forward
        model = MultiResHashSDF.from_npz(npz).to(DEVICE).eval()   # .eval() — режим оцінки
        with torch.no_grad():
            pe = model(torch.from_numpy(eval_pts).to(DEVICE)).cpu().numpy()

        # Обчислюємо F1 окремо для near і unif
        f1n = compute_f1(pe[:n_e], eval_sdf[:n_e] < 0)
        f1u = compute_f1(pe[n_e:], eval_sdf[n_e:] < 0)
        size_kb = npz.stat().st_size / 1024
        rows.append({'id': mid, 'f1_near': f1n, 'f1_unif': f1u, 'size_kb': size_kb})
        del mesh, model, sdf_fn; gc.collect()

    # Агрегована статистика
    eval_df = pd.DataFrame(rows).sort_values('id').reset_index(drop=True)
    print('\n=== Aggregate ===')
    print(f'F1_near   mean={eval_df["f1_near"].mean():.3f}   min={eval_df["f1_near"].min():.3f}   max={eval_df["f1_near"].max():.3f}   target>0.90')
    print(f'F1_unif   mean={eval_df["f1_unif"].mean():.3f}   min={eval_df["f1_unif"].min():.3f}   max={eval_df["f1_unif"].max():.3f}   target>0.95')
    print(f'size      mean={eval_df["size_kb"].mean():.1f} KB  max={eval_df["size_kb"].max():.1f} KB   target<1024 KB')
    eval_df.to_csv(MODELS_DIR / 'eval.csv', index=False)
    eval_df.head(10)
else:
    print('Eval skipped. Set RUN_EVAL_ALL = True to enable.')


## Клітинка 9. Перетренування важких mesh-ів (опціонально)

Читає `eval.csv`, знаходить mesh-і нижче порогу, і перетреновує їх з більшою
хеш-таблицею (T=2^14 замість T=2^13) і вдвічі більшою кількістю ітерацій.

Розмір моделі росте з ~236 КБ до ~470 КБ — все одно у межах 1 МБ.


In [ ]:
if RUN_RETRAIN_HARD:
    eval_df = pd.read_csv(MODELS_DIR / 'eval.csv')         # читаємо результати попередньої оцінки
    F1_NEAR_TGT, F1_UNIF_TGT = 0.92, 0.96                  # пороги з запасом 0.02 над вимогами
    # Знаходимо mesh, у яких хоча б одна метрика нижче порогу
    hard = eval_df[(eval_df['f1_near'] < F1_NEAR_TGT) | (eval_df['f1_unif'] < F1_UNIF_TGT)]
    print(f'retraining {len(hard)} meshes: {hard["id"].tolist()}')

    # Більший варіант конфігурації: log2_table_size=14 → T=16384
    big_cfg = HashConfig(n_levels=8, n_features=2, log2_table_size=14,
                         base_resolution=8, finest_resolution=128, hidden=16)
    log = []; t0_all = time.time()

    for mid in hard['id'].tolist():
        t0 = time.time()
        # n_iters=5000 замість 2500 — більше навчання для важких випадків
        model, history = train_one_mesh(mid, n_iters=5000, cfg=big_cfg, verbose=False)
        out = MODELS_DIR / f'{mid}.npz'
        model.export_npz(out)                              # ПЕРЕЗАПИСУЄ малу модель (вона тепер краща)
        last = history[-1] if history else {'f1_near': float('nan'), 'f1_unif': float('nan')}
        size_kb = out.stat().st_size / 1024
        print(f'[{mid:>2}] {time.time()-t0:5.1f}s  size={size_kb:6.1f} KB  '
              f'f1_near={last["f1_near"]:.3f}  f1_unif={last["f1_unif"]:.3f}')
        log.append({'id': mid, **last, 'size_kb': size_kb})
        del model; gc.collect()
    print(f'\nRetrained {len(log)} meshes in {(time.time()-t0_all)/60:.1f} min')
else:
    print('Retrain skipped. Set RUN_RETRAIN_HARD = True to enable.')


## Клітинка 10. Бенчмарк inference через numba

Завантажуємо кожну навчену модель через `SDFInference` (numba-кернел),
перевіряємо паритет з torch-реалізацією, міряємо швидкість.

Це підтверджує: (1) обидві реалізації дають однакові результати з точністю
до округлень fp16; (2) numba справді швидка — тисячі точок за мс.


In [ ]:
if RUN_BENCH:
    from sdf_inference import SDFInference                 # імпорт всередині клітинки — лише тут потрібний
    rng = np.random.default_rng(123)                       # один seed на весь бенчмарк (відтворюваність)
    rows = []
    for p in mesh_files:
        mid = int(p.stem)
        npz = MODELS_DIR / f'{mid}.npz'
        if not npz.exists(): continue
        nb = SDFInference(npz); nb.warmup()                # тригеримо numba-компіляцію заздалегідь

        # ─── Тест паритету torch vs numba ─────────────────────────────
        torch_m = MultiResHashSDF.from_npz(npz).eval()
        pts = rng.uniform(-1, 1, (1000, 3)).astype(np.float32)
        with torch.no_grad():
            tp = torch_m(torch.from_numpy(pts)).numpy()    # передбачення torch
        npp = nb(pts)                                       # передбачення numba
        max_diff = float(np.abs(tp - npp).max())            # макс. абсолютна різниця

        # ─── Single-point: компільований цикл ─────────────────────────
        ns_single = nb.bench_single_point(100_000)         # нс на запит

        # ─── Batched: для трьох розмірів пакету ───────────────────────
        bench = {}
        for B in [1000, 10_000, 100_000]:
            bp = rng.uniform(-1, 1, (B, 3)).astype(np.float32)
            t = []
            for _ in range(20):                            # 20 повторів — медіана надійніша за середнє
                t0 = time.perf_counter(); _ = nb(bp); t.append(time.perf_counter() - t0)
            bench[B] = float(np.median(t) * 1e3)            # мс

        rows.append({'id': mid,
                     'parity_max_diff': max_diff,
                     'size_kb': npz.stat().st_size / 1024,
                     'ns_single': ns_single,
                     'ms_1k': bench[1000],
                     'ms_10k': bench[10_000],
                     'ms_100k': bench[100_000]})
        del torch_m, nb; gc.collect()

    # ─── Агрегована статистика ────────────────────────────────────────
    bench_df = pd.DataFrame(rows).sort_values('id').reset_index(drop=True)
    print('\n=== Aggregate ===')
    print(f'parity_max_diff   max={bench_df["parity_max_diff"].max():.2e}   target<1e-3')
    print(f'size_kb           mean={bench_df["size_kb"].mean():.0f}  max={bench_df["size_kb"].max():.0f}   target<1024')
    print(f'ns_single         mean={bench_df["ns_single"].mean():.0f}  max={bench_df["ns_single"].max():.0f}   target: few ns')
    print(f'ms_1k    (1 k pts)  mean={bench_df["ms_1k"].mean():.3f}  max={bench_df["ms_1k"].max():.3f}   target: few ms')
    print(f'ms_10k   (10k pts) mean={bench_df["ms_10k"].mean():.3f}  max={bench_df["ms_10k"].max():.3f}')
    print(f'ms_100k (100k pts) mean={bench_df["ms_100k"].mean():.3f}  max={bench_df["ms_100k"].max():.3f}')
    bench_df.to_csv(MODELS_DIR / 'bench.csv', index=False)
    bench_df.head(10)
else:
    print('Bench skipped. Set RUN_BENCH = True to enable.')
